# MLFlow

In [1]:
import mlflow
import os
import dotenv
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

dotenv.load_dotenv(override=True)

ml_client = ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)

c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Set the tracking server

In [2]:
mlflow.set_tracking_uri(os.environ["AZURE_WORKSPACE_MLFLOW_TRACKING_URI"])

Another code friendly way of getting the trackig URI

In [3]:
mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)

print("--> " + mlflow_tracking_uri[:10] + "*************************************************************")

--> azureml://*************************************************************


## Train and track models in MLFlow

In [4]:
import mlflow

mlflow.set_experiment(experiment_name="me-very-own-fecking-experiment")

2026/03/11 14:53:59 INFO mlflow.tracking.fluent: Experiment with name 'me-very-own-fecking-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1773237242268, experiment_id='540691ec-291a-44e3-8b12-9547d36a7de4', last_update_time=None, lifecycle_stage='active', name='me-very-own-fecking-experiment', tags={}, workspace='default'>

In [5]:
df = pd.read_csv("../../data/diabetes-data/diabetes.csv").drop("PatientID", axis=1)

X = df.drop("Diabetic", axis=1)
y = df["Diabetic"]

X_train, X_test, y_train, y_test = train_test_split(X.to_numpy(), y.to_numpy(), test_size=0.3, random_state=42)


### Enable autologging

In [6]:
from xgboost import XGBClassifier

with mlflow.start_run():
    mlflow.xgboost.autolog()

    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:54:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/03/11 14:54:34 ERROR mlflow.xgboost: Failed to log feature importance plot. XGBoost autologging will ignore the failure and continue. Exception: 
Traceback (most recent call last):
  File "c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages\mlflow\xgboost\__init__.py", line 824, in train_impl
    log_feature_importance_plot(features, importance, imp_type)
  File "c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages\mlflow\xgboost\__init__.py", line 674, in log_feature_importance_plot
    mlflow.log_artifact(filepath)
  File "c:\Users\dgouwy\Documents\devoProjects\azure-machine-learning\.venv\Lib\site-packages\mlflow\trac

🏃 View run bright_oven_jf0pvyn5 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/540691ec-291a-44e3-8b12-9547d36a7de4/runs/6bfdd074-0f58-46fb-8b14-64559c9b0ae0
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourceGroups/dg-associate-data-science/providers/Microsoft.MachineLearningServices/workspaces/aml-dg-associate-data-science/#/experiments/540691ec-291a-44e3-8b12-9547d36a7de4


### Custom logging